# MedImageParse 3D Fine-tuning Pipeline

This notebook demonstrates a complete end-to-end workflow for fine-tuning the MedImageParse3D model using Azure Machine Learning. The pipeline includes:

1. **Data Management**: Registering pre-processed training data and configuration files in Azure ML
2. **Pipeline Execution**: Running the fine-tuning job on Azure ML compute
3. **Model Deployment**: Deploying the fine-tuned model as a managed online endpoint
4. **Validation**: Testing the deployed model with sample data

## How it works
The fine-tuning pipeline operates on **2D PNG slices** that have been pre-processed from 3D volumetric data (NIfTI/NPZ). Each dataset (e.g. PET, MR_crossmoda) has a `processed/` directory containing:
- `train/` — 2D PNG image slices
- `train_mask/` — corresponding 2D segmentation masks (PNG)
- `train.json` — annotation manifest with class prompts

The Hydra-based training config (`MIP3D_finetune.yaml`) composes the full pipeline: BiomedParse model with FocalNet backbone + BoltzFormer decoder, AdamW optimizer, PyTorch Lightning trainer, and custom dataset/loss/evaluator modules.

## Prerequisites
- Azure ML workspace configured (with `config.json`)
- Training data pre-processed into the expected directory layout (see below)
- GPU compute cluster with H100 or A100 GPUs


## Setup and Authentication

This section handles the initial setup including:
- Importing required Azure ML SDK components and other dependencies
- Establishing authentication with Azure using DefaultAzureCredential
- Creating the MLClient instance for interacting with Azure ML services

create a config.json file in the following format:
```
{
       "subscription_id": "my-subscription-id",
       "resource_group": "my-resourcegroup-name",
       "workspace_name": "my-workspace-name"
   }
```

In [ ]:
from azure.ai.ml import MLClient, load_component
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data, Environment, BuildContext, AmlCompute, Model
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.dsl import pipeline
import random
import os
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
)
import json
import base64
import matplotlib.pyplot as plt
import numpy as np
from azure.core.exceptions import ResourceNotFoundError

credentials = DefaultAzureCredential()
ml_client = MLClient.from_config(credentials)
ml_registry = MLClient(credential=credentials, registry_name="azureml")


# Upload Data

This section creates and registers data assets in Azure ML that will be used for fine-tuning.

## Data Layout

The Hydra config (`biomedparse_finetune_dataset.yaml`) expects the data input path to be a **parent directory** containing `data/MIP3D_CVPR_FT/`. Under that, each sub-dataset has a `processed/` directory with 2D PNG slices:

```
<data_root>/
└── data/
    └── MIP3D_CVPR_FT/
        ├── class_prompts.json
        ├── PET/
        │   ├── train/                (raw .npz volumes — not used by training directly)
        │   ├── test/                 (raw .npz test volumes)
        │   ├── test_gt/             (raw .npz test masks)
        │   └── processed/
        │       ├── train/           (2D PNG image slices)
        │       ├── train_mask/      (2D PNG segmentation masks)
        │       └── train.json       (annotations + class_prompts)
        └── MR_crossmoda/
            ├── train/               (raw .npz volumes)
            ├── test/                (raw .npz test volumes)
            ├── test_gt/            (raw .npz test masks)
            └── processed/
                ├── train/          (2D PNG image slices)
                ├── train_mask/     (2D PNG segmentation masks)
                └── train.json      (annotations + class_prompts)
```

Each `train.json` contains:
- `class_prompts`: mapping of class IDs to text prompt variants (e.g. "Lesion in whole body PET")
- `annotations`: list of `{file_name, mask_file, split, class_ids, instance_label}` entries

The `BiomedSegDataset` class reads images from `processed/train/`, masks from `processed/train_mask/`, and metadata from `processed/train.json`.

## Important: Path Structure

The Hydra dataset config references paths as `${mounts.external}/data/MIP3D_CVPR_FT/{dataset}/processed`. Since `EXTERNAL` is set to the `--data` input mount, **the data asset must point to the directory that _contains_ `data/MIP3D_CVPR_FT/`** — not the `MIP3D_CVPR_FT` folder directly.

- **parameters**: Configuration file (parameters.yaml) that overrides default training hyperparameters

⚠️ **Note**: The default Hydra config (`MIP3D_finetune.yaml`) has `# - parameters` commented out. To use your `parameters.yaml` overrides, you need to uncomment that line in `MIP3D_finetune.yaml` or pass overrides via Hydra command-line arguments.


In [ ]:
name = "medimageparse3d"

# Option A: Reference existing data on a datastore.
# The Hydra config expects EXTERNAL to point to a directory containing
# data/MIP3D_CVPR_FT/. If your datastore already has this structure,
# you can reference it directly instead of uploading:
#
# training_data_uri = "azureml://datastores/olympus_external/paths/"
#
# Then pass training_data_uri as the data input to the pipeline below.

# Option B: Upload a local directory that has the data/MIP3D_CVPR_FT/ structure.
# The path below should be the PARENT directory that contains data/MIP3D_CVPR_FT/.
training_data = Data(
    path="/datadrive/olympus-external",  # Contains data/MIP3D_CVPR_FT/{PET,MR_crossmoda}/processed/
    type=AssetTypes.URI_FOLDER,
    description=f"{name} training data root (contains data/MIP3D_CVPR_FT/)",
    name=f"{name}-training_data",
)
training_data = ml_client.data.create_or_update(training_data)
print(
    f"Data asset training_data created or updated.",
    training_data.name,
    training_data.version,
)

# Create and upload parameters file
parameters = Data(
    path="parameters.yaml",
    type=AssetTypes.URI_FILE,
    description=f"{name} parameters",
    name=f"{name}-parameters",
)
parameters = ml_client.data.create_or_update(parameters)
print(
    f"Data asset parameters created or updated.",
    parameters.name,
    parameters.version,
)

# Store references for later use
data_assets = {
    "training_data": training_data,
    "parameters": parameters,
}


# Create and run Pipeline

## Select / Create Compute

This section manages the compute resources needed for fine-tuning:

- **GPU Compute**: Uses Standard_NC40ads_H100_v5 (4x A100 80GB) or Standard_NC40ads_H100_v5 (H100) instances
- **Conditional Creation**: Only creates new compute if it doesn't already exist
- **Resource Management**: Automatically scales down when not in use to minimize costs

The default trainer config uses `bf16-mixed` precision with DDP (DistributedDataParallel) strategy. The default batch size is 1 per GPU (overridden to 4 in parameters.yaml).


In [ ]:
compute_name = "gpucluster"

try:
    compute = ml_client.compute.get(name=compute_name)
except ResourceNotFoundError:
    compute_cluster = AmlCompute(
        name=compute_name,
        description="GPU cluster compute for fine-tuning",
        min_instances=1,
        max_instances=2,
        size="Standard_NC40ads_H100_v5",
    )
    compute = ml_client.compute.begin_create_or_update(compute_cluster).result()

## Pipeline Setup and Execution

This section handles the core pipeline creation and execution:

1. **Component Loading**: Retrieves the `medimageparse_3d_finetune` component from the AzureML registry
2. **Pre-trained Model**: Fetches the latest MedImageParse3D model (with BiomedParse v2 checkpoint)
3. **Pipeline Definition**: Creates a pipeline that maps inputs (model, data, config) to outputs (MLflow model)
4. **Job Submission**: Submits the pipeline job to Azure ML with experiment tracking

### Hydra Config Chain
The component's entrypoint (`medimageparse_3d_finetune.py`) launches Olympus training via Hydra. The config name is `MIP3D_finetune` which composes:
- **model**: `biomedparse` (FocalNet backbone + BoltzFormer decoder)
- **datamodule**: `biomedparse_finetune_datamodule` → `BiomedSegDataset` reading from `${EXTERNAL}/data/MIP3D_CVPR_FT/`
- **trainer**: PyTorch Lightning with bf16-mixed, DDP strategy
- **optimizer**: AdamW (lr=0.00002, weight_decay=0.01)
- **checkpoint loader**: loads `biomedparse_v2.ckpt` from model input
- **loss**: SEEM loss with focal classification and MedSAM segmentation losses


In [ ]:
# Get the pipeline component
finetune_pipeline_component = ml_registry.components.get(
    name="medimageparse_3d_finetune", label="latest"
)
print(
    "Component loaded",
    finetune_pipeline_component.name,
    finetune_pipeline_component.version,
)

# Get the latest MIP model
model = ml_registry.models.get(name="MedImageParse3D", label="latest")


@pipeline(name="medimageparse_3d_finetuning" + str(random.randint(0, 100000)))
def create_pipeline():
    mip_pipeline = finetune_pipeline_component(
        pretrained_mlflow_model=model.id,
        data=data_assets["training_data"].id,
        config=data_assets["parameters"].id,
    )
    return {"mlflow_model_folder": mip_pipeline.outputs.mlflow_model_folder}


pipeline_object = create_pipeline()
pipeline_object.compute = compute.name
pipeline_object.settings.continue_on_step_failure = False
pipeline_job = ml_client.jobs.create_or_update(
    pipeline_object, experiment_name="medimageparse_3d_finetune_experiment"
)
pipeline_job_run_id = pipeline_job.name
pipeline_job

### Monitor Pipeline Job

This cell displays the pipeline job details including:
- Job status and progress
- Links to view the job in Azure ML Studio
- Job ID for future reference
- Real-time updates on training progress

You can click on the Studio link to monitor training metrics, logs, and resource utilization in real-time.

# Deploy Fine-tuned Model

This section handles the deployment of the fine-tuned 3D model as a managed online endpoint:

## Key Steps:
1. **Model Registration**: Register the fine-tuned model from the pipeline output
2. **Endpoint Creation**: Create a managed online endpoint for real-time inference
3. **Deployment Configuration**: Set up the deployment with appropriate compute resources
4. **Resource Allocation**: Use GPU instances for optimal inference performance

The deployment creates a REST API endpoint that can be called for real-time 3D medical image segmentation and analysis.


In [ ]:
# Run this cell if you reconnected to the notebook.

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credentials = DefaultAzureCredential()
ml_client = MLClient.from_config(credentials)

if "pipeline_job_run_id" not in locals():
    ##  Retrieved by checking the json of the parent job in AzureML studio (under "See all properties") or in output of the cell where you started the job under "Name".
    pipeline_job_run_id = ""

if not len(pipeline_job_run_id):
    raise ValueError(
        "Your kernel may have died! You need to set the pipeline_job_run_id manually from the output above (job name), or from the experiment in AzureML portal."
    )

pipeline_job = ml_client.jobs.get(name=pipeline_job_run_id)

## Model Registration and Endpoint Deployment

This section performs the actual model registration and endpoint deployment:

### Model Registration:
- **Output Path**: References the pipeline job output containing the fine-tuned model
- **MLflow Format**: Uses MLflow model format for easy deployment and versioning
- **Unique Naming**: Creates a unique model name combining project name and job ID

### Endpoint Creation:
- **Managed Endpoint**: Creates a managed online endpoint for easy scaling and management
- **GPU Deployment**: Uses Standard_NC40ads_H100_v5 for optimal inference performance. Standard_NC24ads_A100_v4 would also work.
- **Single Instance**: Starts with 1 instance, can be scaled up based on demand

⚠️ **Note**: This process can take 10-15 minutes to complete as Azure provisions the compute resources and deploys the model.


In [ ]:
run_model = Model(
    path=f"azureml://jobs/{pipeline_job.name}/outputs/mlflow_model_folder",
    name=f"MIP3D-{name}-{pipeline_job.name}",
    description="MedImageParse3D model created from fine-tuning run.",
    type=AssetTypes.MLFLOW_MODEL,
)

# Register the Model
run_model = ml_client.models.create_or_update(run_model)

# Create endpoint and deployment with the fine-tuned 3D model
endpoint = ManagedOnlineEndpoint(name=name)
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
deployment = ManagedOnlineDeployment(
    name=name,
    endpoint_name=endpoint.name,
    model=run_model.id,
    instance_type="Standard_NC40ads_H100_v5",
    instance_count=1,
)
deployment = ml_client.online_deployments.begin_create_or_update(deployment).result()


# Validate Deployment

This section tests the deployed 3D model to ensure it's working correctly:

## Test Process:
1. **Sample Data Preparation**: Load a test image (PNG slice from the processed training data)
2. **Request Formatting**: Convert the image to base64 encoding and create a properly formatted JSON request
3. **Text Prompt**: Include a text description to guide the model's segmentation task (matching the class_prompts from training)
4. **API Call**: Send the request to the deployed endpoint
5. **Results Visualization**: Display the original image alongside the generated segmentation masks

Note: For 3D inference with NIfTI volumes, see the `mip3d-deploy.ipynb` notebook. Here we validate with 2D slices from the training dataset.


In [ ]:
def read_image(image_path):
    """Read an image file as raw bytes for base64 encoding."""
    with open(image_path, "rb") as f:
        return f.read()


# Use a sample image from the processed training data
# The processed/ directory contains 2D PNG slices extracted from 3D volumes
data_root = "/datadrive/olympus-external/data/MIP3D_CVPR_FT"
processed_dir = os.path.join(data_root, "PET", "processed", "train")

# Pick the first available image
sample_images = sorted([f for f in os.listdir(processed_dir) if f.endswith(".png")])
sample_image_path = os.path.join(processed_dir, sample_images[0])
print(f"Using sample image: {sample_images[0]}")

# Use a text prompt matching the PET class prompts
text_prompt = "Lesion in whole body PET"

data = {
    "input_data": {
        "columns": ["image", "text"],
        "index": [0],
        "data": [
            [
                base64.encodebytes(read_image(sample_image_path)).decode("utf-8"),
                text_prompt,
            ]
        ],
    }
}
data_json = json.dumps(data)

# Create request json
request_file_name = "sample_request_data.json"
with open(request_file_name, "w") as request_file:
    json.dump(data, request_file)


In [ ]:
def decode_json_to_array(json_encoded):
    """Decode an image pixel data array in JSON.
    Return image pixel data as an array.
    """
    array_metadata = json.loads(json_encoded)
    base64_encoded = array_metadata["data"]
    shape = tuple(array_metadata["shape"])
    dtype = np.dtype(array_metadata["dtype"])
    array_bytes = base64.b64decode(base64_encoded)
    array = np.frombuffer(array_bytes, dtype=dtype).reshape(shape)
    return array


def plot_segmentation_masks(original_image, segmentation_masks, labels=None):
    """Plot segmentation masks overlaid on the original image."""
    fig, ax = plt.subplots(1, len(segmentation_masks) + 1, figsize=(5 * (len(segmentation_masks) + 1), 5))
    if len(segmentation_masks) == 0:
        ax = [ax]
    ax[0].imshow(original_image)
    ax[0].set_title("Original Image")
    ax[0].axis("off")

    for i, mask in enumerate(segmentation_masks):
        ax[i + 1].imshow(original_image)
        title = labels[i] if labels and i < len(labels) else f"Mask {i+1}"
        ax[i + 1].set_title(title)
        overlay = np.zeros((*mask.shape[:2], 4), dtype=np.float32)
        overlay[mask > 0] = [1.0, 0.0, 0.0, 0.6]
        ax[i + 1].imshow(overlay)
        ax[i + 1].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
response = ml_client.online_endpoints.invoke(
    endpoint_name=name,
    deployment_name=name,
    request_file=request_file_name,
)
result_list = json.loads(response)
image_features_str = result_list[0]["image_features"]
image_features = decode_json_to_array(image_features_str)
text_features = result_list[0].get("text_features", [])

# Load the original image for visualization
from PIL import Image
original_image = np.array(Image.open(sample_image_path))
plot_segmentation_masks(original_image, image_features, labels=text_features)


In [ ]:
_ = ml_client.online_endpoints.begin_delete(name=name).wait()
